# Master Audio Paper Pipeline — Frozen Artifact Replay

This notebook validates the current audio-only paper-safe checkpoint from `machine_learning_audio`.

Mode used here: **Mode A: Frozen Artifact Replay**.

Purpose:
- validate final paper metrics without rebuilding all upstream models;
- inspect report and confusion matrix artifacts;
- export a compact validation summary for the conference paper.

This notebook does **not** retrain from raw audio. Full rebuild is Mode B and can drift across package versions, random seeds, and thread scheduling.

In [1]:
from pathlib import Path
import json
import platform
import importlib
import numpy as np
import pandas as pd

CWD = Path.cwd()
if CWD.name == "machine_learning_audio":
    AUDIO_DIR = CWD
    ROOT = CWD.parent
else:
    ROOT = CWD
    AUDIO_DIR = ROOT / "machine_learning_audio"
assert AUDIO_DIR.exists(), f"Missing {AUDIO_DIR}"

print("ROOT:", ROOT)
print("AUDIO_DIR:", AUDIO_DIR)
print("Python:", platform.python_version())

for name in ["numpy", "pandas", "sklearn", "joblib"]:
    try:
        module = importlib.import_module(name)
        print(f"{name}: {getattr(module, '__version__', 'OK')}")
    except Exception as exc:
        print(f"{name}: MISSING {exc}")

ROOT: C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual
AUDIO_DIR: C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio
Python: 3.13.9
numpy: 2.4.4
pandas: 2.3.3


sklearn: 1.8.0
joblib: 1.5.3


## 1. Locate Frozen Checkpoint Artifacts

The checkpoint README states that the final paper-safe run slug is `audio_lift_source_blend_select`.

In [2]:
RUN = "audio_lift_source_blend_select"
paths = {
    "report": AUDIO_DIR / f"{RUN}_final_test_report.csv",
    "confusion": AUDIO_DIR / f"{RUN}_final_test_confusion_matrix.csv",
    "leaderboard": AUDIO_DIR / f"{RUN}_oof_leaderboard.csv",
    "selected": AUDIO_DIR / f"{RUN}_selected_without_test.json",
    "method_card": AUDIO_DIR / f"{RUN}_method_card_before_test.json",
    "protocol_summary": AUDIO_DIR / f"{RUN}_protocol_summary.json",
}

for key, path in paths.items():
    print(f"{key:16s}", "OK" if path.exists() else "MISSING", path)

missing = [key for key, path in paths.items() if not path.exists()]
assert not missing, f"Missing checkpoint artifacts: {missing}"

report           OK C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\audio_lift_source_blend_select_final_test_report.csv
confusion        OK C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\audio_lift_source_blend_select_final_test_confusion_matrix.csv
leaderboard      OK C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\audio_lift_source_blend_select_oof_leaderboard.csv
selected         OK C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\audio_lift_source_blend_select_selected_without_test.json
method_card      OK C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\audio_lift_source_blend_select_method_card_before_test.json
protocol_summary OK C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\audio_lift_source_blend_select_protocol_summary.j

## 2. Load Locked Metadata

In [3]:
selected = json.loads(paths["selected"].read_text(encoding="utf-8"))
method_card = json.loads(paths["method_card"].read_text(encoding="utf-8"))
protocol_summary = json.loads(paths["protocol_summary"].read_text(encoding="utf-8"))

print("Selected lock:")
print(json.dumps(selected, indent=2, ensure_ascii=False)[:3000])
print("\nMethod card:")
print(json.dumps(method_card, indent=2, ensure_ascii=False)[:3000])

Selected lock:
{
  "selection_rule": "highest train-only OOF macro/contact score over lift-anchor source blends",
  "selected_without_test": {
    "recipe_kind": "audio_lift_source_blend",
    "source_name": "report_gate_onehot",
    "source_weight": 0.05,
    "blend_mode": "segment_lift",
    "macro_f1": 0.9353447727801658,
    "contact_macro_f1": 0.9164944534943943,
    "binary_macro_f1": 0.99077894279929,
    "selection_score": 0.9352330939963467
  },
  "method_card": "/home/ttung05/Desktop/tree_audio/outputs/audio_feature_benchmarks/audio_lift_source_blend_select/reports/audio_lift_source_blend_select_method_card_before_test.json",
  "leaderboard_path": "/home/ttung05/Desktop/tree_audio/outputs/audio_feature_benchmarks/audio_lift_source_blend_select/reports/audio_lift_source_blend_select_oof_leaderboard.csv"
}

Method card:
{
  "protocol": "audio_only_lift_source_blend_no_test_until_lock",
  "allowed_selection_data": "hand/default train labels and locked audio-only OOF probabilitie

## 3. Validate Final Report Metrics

In [4]:
report = pd.read_csv(paths["report"])
display(report)

row = report.iloc[0].to_dict()
expected = {
    "accuracy_4class": 0.7967552951780081,
    "macro_f1_4class": 0.7026719927789447,
    "contact_macro_f1": 0.625050260344378,
    "binary_macro_f1": 0.9291164642187257,
}

print("Metric checks:")
for key, exp in expected.items():
    got = float(row[key])
    ok = abs(got - exp) < 1e-12
    print(f"{key:20s} got={got:.15f} expected={exp:.15f} ok={ok}")
    assert ok, f"{key} mismatch: got {got}, expected {exp}"

print("PASS: final report metrics match expected checkpoint values.")

,feature_set,feature_name,n_features,split,model,status,accuracy_4class,macro_precision_4class,macro_recall_4class,macro_f1_4class,...,twig_recall,twig_f1,twig_support,selected_by,selected_score,selected_oof_macro_f1,selected_oof_contact_macro_f1,selected_source_name,selected_source_weight,selected_blend_mode
0,total240,Total 240D,240,robot_test_final,audio_lift_source_blend,ok,0.796755,0.770801,0.721682,0.702672,...,0.585586,0.577778,333,train_only_oof_audio_lift_source_blend,0.935233,0.935345,0.916494,report_gate_onehot,0.05,segment_lift


Metric checks:
accuracy_4class      got=0.796755295178008 expected=0.796755295178008 ok=True
macro_f1_4class      got=0.702671992778945 expected=0.702671992778945 ok=True
contact_macro_f1     got=0.625050260344378 expected=0.625050260344378 ok=True
binary_macro_f1      got=0.929116464218726 expected=0.929116464218726 ok=True
PASS: final report metrics match expected checkpoint values.


## 4. Validate Confusion Matrix and Recompute Metrics

In [5]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

cm = pd.read_csv(paths["confusion"], index_col=0)
display(cm)

labels = ["ambient", "leaf", "trunk", "twig"]
contact_labels = [1, 2, 3]

y_true = []
y_pred = []
for i, true_label in enumerate(labels):
    for j, pred_label in enumerate(labels):
        count = int(cm.loc[true_label, pred_label])
        y_true.extend([i] * count)
        y_pred.extend([j] * count)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc = accuracy_score(y_true, y_pred)
macro = f1_score(y_true, y_pred, labels=[0,1,2,3], average="macro", zero_division=0)
contact = f1_score(y_true, y_pred, labels=contact_labels, average="macro", zero_division=0)
binary = f1_score((y_true > 0).astype(int), (y_pred > 0).astype(int), labels=[0,1], average="macro", zero_division=0)

checks = {
    "accuracy_4class": acc,
    "macro_f1_4class": macro,
    "contact_macro_f1": contact,
    "binary_macro_f1": binary,
}

for key, got in checks.items():
    exp = expected[key]
    ok = abs(got - exp) < 1e-12
    print(f"{key:20s} recomputed={got:.15f} expected={exp:.15f} ok={ok}")
    assert ok, f"Recomputed {key} mismatch"

pr, rc, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0,1,2,3], zero_division=0)
per_class = pd.DataFrame({"class": labels, "precision": pr, "recall": rc, "f1": f1, "support": support})
display(per_class)

print("PASS: confusion matrix recomputes final checkpoint metrics.")

,ambient,leaf,trunk,twig
ambient,1132,0,0,0
leaf,2,277,0,14
trunk,126,38,164,133
twig,28,106,4,195


accuracy_4class      recomputed=0.796755295178008 expected=0.796755295178008 ok=True
macro_f1_4class      recomputed=0.702671992778945 expected=0.702671992778945 ok=True
contact_macro_f1     recomputed=0.625050260344378 expected=0.625050260344378 ok=True
binary_macro_f1      recomputed=0.929116464218726 expected=0.929116464218726 ok=True


,class,precision,recall,f1,support
0,ambient,0.878882,1.000000,0.935537,1132
1,leaf,0.657957,0.945392,0.775910,293
2,trunk,0.976190,0.355748,0.521463,461
3,twig,0.570175,0.585586,0.577778,333


PASS: confusion matrix recomputes final checkpoint metrics.


## 5. OOF Leaderboard and Selection Evidence

In [6]:
leaderboard = pd.read_csv(paths["leaderboard"])
print("Leaderboard shape:", leaderboard.shape)
display(leaderboard.head(10))

if "selection_score" in leaderboard.columns:
    display(leaderboard.sort_values("selection_score", ascending=False).head(10))

Leaderboard shape: (156, 8)


,recipe_kind,source_name,source_weight,blend_mode,macro_f1,contact_macro_f1,binary_macro_f1,selection_score
0,audio_lift_source_blend,report_gate_onehot,0.05,segment_lift,0.935345,0.916494,0.990779,0.935233
1,audio_lift_source_blend,report_gate_onehot,0.10,segment_lift,0.935345,0.916494,0.990779,0.935233
2,audio_lift_source_blend,report_gate_onehot,0.15,segment_lift,0.935345,0.916494,0.990779,0.935233
3,audio_lift_source_blend,report_gate_onehot,0.20,segment_lift,0.935345,0.916494,0.990779,0.935233
4,audio_lift_source_blend,report_gate_onehot,0.25,segment_lift,0.935345,0.916494,0.990779,0.935233
5,audio_lift_source_blend,report_gate_onehot,0.33,segment_lift,0.935345,0.916494,0.990779,0.935233
6,audio_lift_source_blend,report_gate_onehot,0.50,segment_lift,0.932081,0.912143,0.990779,0.931969
7,audio_lift_source_blend,mfcc40_grid,0.05,segment_lift,0.931235,0.911014,0.990779,0.931123
8,audio_lift_source_blend,mfcc40_grid,0.10,segment_lift,0.931235,0.911014,0.990779,0.931123
9,audio_lift_source_blend,mfcc40_grid,0.15,segment_lift,0.931235,0.911014,0.990779,0.931123


,recipe_kind,source_name,source_weight,blend_mode,macro_f1,contact_macro_f1,binary_macro_f1,selection_score
0,audio_lift_source_blend,report_gate_onehot,0.05,segment_lift,0.935345,0.916494,0.990779,0.935233
1,audio_lift_source_blend,report_gate_onehot,0.10,segment_lift,0.935345,0.916494,0.990779,0.935233
2,audio_lift_source_blend,report_gate_onehot,0.15,segment_lift,0.935345,0.916494,0.990779,0.935233
3,audio_lift_source_blend,report_gate_onehot,0.20,segment_lift,0.935345,0.916494,0.990779,0.935233
4,audio_lift_source_blend,report_gate_onehot,0.25,segment_lift,0.935345,0.916494,0.990779,0.935233
5,audio_lift_source_blend,report_gate_onehot,0.33,segment_lift,0.935345,0.916494,0.990779,0.935233
6,audio_lift_source_blend,report_gate_onehot,0.50,segment_lift,0.932081,0.912143,0.990779,0.931969
7,audio_lift_source_blend,mfcc40_grid,0.05,segment_lift,0.931235,0.911014,0.990779,0.931123
8,audio_lift_source_blend,mfcc40_grid,0.10,segment_lift,0.931235,0.911014,0.990779,0.931123
9,audio_lift_source_blend,mfcc40_grid,0.15,segment_lift,0.931235,0.911014,0.990779,0.931123


## 6. Export Paper Validation Summary

In [7]:
summary = {
    "mode": "Frozen Artifact Replay",
    "run_slug": RUN,
    "split": row.get("split", "robot_test_final"),
    "audio_only": True,
    "uses_image_or_multimodal_features": False,
    "metrics": {key: float(row[key]) for key in expected},
    "n_samples_from_confusion_matrix": int(cm.to_numpy().sum()),
    "selected_lock": selected,
    "method_card": method_card,
}

out_json = AUDIO_DIR / "00_master_audio_paper_pipeline_validation_summary.json"
out_csv = AUDIO_DIR / "00_master_audio_paper_pipeline_per_class_metrics.csv"
Path(out_json).write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
per_class.to_csv(out_csv, index=False)

print("Wrote:", out_json)
print("Wrote:", out_csv)
print("VALIDATION PASS")

Wrote: C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\00_master_audio_paper_pipeline_validation_summary.json
Wrote: C:\Users\ADMIN\OneDrive\Documents\_Project\multi_model_audio_visual\machine_learning_audio\00_master_audio_paper_pipeline_per_class_metrics.csv
VALIDATION PASS
